##### Primeira modelagem (tentar refazer em casa com matematica simbolica)

In [134]:
#funcao objetivo (obj: maximizar)
#a melhor estrutura que achamos foi a lambda, ficaria:
#f = 300x1 + 500x2 , daria pra usar matematica simbolica

fo = lambda x1, x2: 300 * x1 + 500  * x2

#restricoes (obj: delimitar areas de solucao da PO)
#restricoes e uma lista de dicionarios

restricoes = [
    {"alpha":2, "beta":1, "gamma":16}, #restricao 1 
    {"alpha":1, "beta":2, "gamma":11}, #restricao 2
    {"alpha":1, "beta":3, "gamma":15}  #restricao 3
    #restricao n
]


In [135]:
retas = []

for restricao in restricoes:
    a = -restricao["alpha"] / restricao["beta"]
    b = restricao["gamma"] / restricao["beta"]
    print(a,b) #esses sao os coeficinetes das retas das restricoes 
    retas.append(  # cria uma lista de dicionarios com os coeficientes
        {"a":a, "b":b}
    )
    
print(retas)

-2.0 16.0
-0.5 5.5
-0.3333333333333333 5.0
[{'a': -2.0, 'b': 16.0}, {'a': -0.5, 'b': 5.5}, {'a': -0.3333333333333333, 'b': 5.0}]


In [136]:
#encontrar onde cada reta cruza com cada eixo, dividindo em (PV ponto venrtical) e (PH ponto horizontal)
vertices = []

for reta in retas:
    pv = (0,reta["b"])
    ph = (- reta["b"] / reta["a"],0)
    print(pv, ph)
    vertices = vertices + [pv,ph] #juntei duas listas ja existentes numa nova lista chamada vertices
    #sendo uma lista de tuplas
print(vertices)



(0, 16.0) (8.0, 0)
(0, 5.5) (11.0, 0)
(0, 5.0) (15.0, 0)
[(0, 16.0), (8.0, 0), (0, 5.5), (11.0, 0), (0, 5.0), (15.0, 0)]


In [137]:
#igualar as equacoes pra achar onde elas se encontram
#vamos fazer isso com o combinations

from itertools import combinations

l = ["a", "b", "c"] #teste de uma lista qualquer so pra mostrar como que o cobinations funciona
print(list(combinations(l,2))) #tem que usar o list pra transformar essa combinacao em lista, se nao fica
#so no mundo das ideias, tipo a funcao range()

#agora com o exdemplo verdadeiro usando as retas
combinacoes_retas = list(combinations(retas,2))
display(combinacoes_retas)

[('a', 'b'), ('a', 'c'), ('b', 'c')]


[({'a': -2.0, 'b': 16.0}, {'a': -0.5, 'b': 5.5}),
 ({'a': -2.0, 'b': 16.0}, {'a': -0.3333333333333333, 'b': 5.0}),
 ({'a': -0.5, 'b': 5.5}, {'a': -0.3333333333333333, 'b': 5.0})]

In [138]:
for dict_reta1, dict_reta2 in combinacoes_retas:
    a1 = dict_reta1["a"]
    b1 = dict_reta1["b"]
    a2 = dict_reta2["a"]
    b2 = dict_reta2["b"]

    x1_cruzamento = (b2 - b1)/(a1 - a2)
    x2_cruzamento = a1 * x1_cruzamento + b1

    vertices.append((x1_cruzamento, x2_cruzamento))
display(vertices)    

[(0, 16.0),
 (8.0, 0),
 (0, 5.5),
 (11.0, 0),
 (0, 5.0),
 (15.0, 0),
 (7.0, 2.0),
 (6.6, 2.8000000000000007),
 (2.9999999999999996, 4.0)]

In [139]:
#garantir que os pontos estejam dentro das restricoes 

checa_restricao = lambda alpha, x1, beta, x2, gamma: \
    alpha * x1 + beta * x2 <= gamma #isso aqui pode gerar problema, arranjar outro jeito de arrumar




In [140]:
x1, x2 = vertices[0]

#maneira mais gorila de fazer 
c1 = checa_restricao(restricoes[0]["alpha"], x1, 
                     restricoes[0]["beta"], x2, 
                     restricoes[0]["gamma"])

#maneira mais inteligente de fazer, usa uma tecnica nova que ainda nao aprendemos 
c2 = checa_restricao(**restricoes[1], x1=x1, x2=x2)

#maneira mais inteligente de fazer, usa uma tecnica nova que ainda nao aprendemos 
c3 = checa_restricao(**restricoes[2], x1=x1, x2=x2)

print(c1 and c2 and c3) #faz uso dos operadores logicos (AND) e retorna o valor do resultado logico

False


In [141]:
vertices_finais = []

for vertice in vertices:
    c = True
    for restricao in restricoes:
        checagem_atual = checa_restricao(**restricao, x1=vertice[0], x2=vertice[1])
        c = c and checagem_atual
    if c == True:
        vertices_finais.append(vertice)


In [142]:
print(vertices_finais)

[(8.0, 0), (0, 5.0), (7.0, 2.0), (2.9999999999999996, 4.0)]


In [143]:
solucao_final = None
valor_otimo = float("-inf")
for vertice in vertices_finais:
    valor = fo(x1=vertice[0], x2=vertice[1])
    if valor > valor_otimo:
        solucao_final = vertice
        valor_otimo = valor
    
print(f"solucao final: {solucao_final} {valor_otimo}")

solucao final: (7.0, 2.0) 3100.0


In [ ]:
#funcao final #o gamma correto tem 2 m, nessa celula ta com uma so
from typing import Callable
def po_vertices(fo: Callable,
                restricoes: list[dict],
                tipo: str) -> tuple:
    retas = []
    for dict_restricao in restricoes:
        a = - dict_restricao["alpha"] / dict_restricao["beta"]
        b = dict_restricao["gama"] / dict_restricao["beta"]
        retas.append(
            {"a": a, "b": b}
        )
    
    vertices = []
    for reta in retas:
        x1 = - reta["b"] / reta["a"]
        x2 = reta["b"]
        pv = (0, x2)
        ph = (x1, 0)
        vertices = vertices + [pv, ph]
    
    combinacoes_retas = list(combinations(retas, 2))
    for dict_reta_um, dict_reta_dois in combinacoes_retas:
        a1 = dict_reta_um["a"]
        b1 = dict_reta_um["b"]
        a2 = dict_reta_dois["a"]
        b2 = dict_reta_dois["b"]

        x1_cruzamento = (b2 - b1) / (a1 - a2)
        x2_cruzamento = a1 * x1_cruzamento + b1

        vertices.append((x1_cruzamento, x2_cruzamento))
    
    if tipo == "max":
        checa_restricao = lambda alpha, x1, beta, x2, gama: \
            abs((alpha * x1 + beta * x2) - gama) <= 0.0001 or \
            alpha * x1 + beta * x2 <= gama
    elif tipo == "min":
        checa_restricao = lambda alpha, x1, beta, x2, gama: \
            abs((alpha * x1 + beta * x2) - gama) <= 0.0001 or \
            alpha * x1 + beta * x2 >= gama
    else:
        raise ValueError("O tipo deve ser max ou min")

    vertices_finais = []
    for i, vertice in enumerate(vertices, 1):
        c = True
        for j, restricao in enumerate(restricoes, 1):
            checagem_atual = checa_restricao(**restricao, 
                                            x1=vertice[0], 
                                            x2=vertice[1])
            c = c and checagem_atual
        
        if c == True:
            vertices_finais.append(vertice)
        
    solucao_final = None
    valor_otimo = float("-inf") if tipo == "max" else float("inf")
    for vertice in vertices_finais:
        valor = fo(x1=vertice[0], x2=vertice[1])
        if valor > valor_otimo and tipo == "max":
            solucao_final = vertice
            valor_otimo = valor
        elif valor < valor_otimo and tipo == "min":
            solucao_final = vertice
            valor_otimo = valor
    
    return solucao_final, valor_otimo
    

In [146]:
#chamar a funcao, dar os parametros e testar pra ver se funciona